<a href="https://colab.research.google.com/github/saulo-albuquerque-phys/GWgpu_jax/blob/prior_definitions/examples/Parameter_Estimation_ripplegw_PP_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GWgpu_jax — **PP (probability–probability) test** with IMRPhenomD injections

This notebook validates the GWgpu_jax parameter-estimation pipeline end-to-end
by running a **PP test**:

1. Draw **N injections** of source parameters **directly from the prior**
   (so distance is volumetric → a wide spread of network SNRs, satisfying the
   "different SNR values" requirement).
2. For each injection: inject an **IMRPhenomD** signal into coloured Gaussian
   noise, run the **two-phase nested sampler**, and save the **posterior
   samples** + a **corner plot** into its own ID-tagged folder under
   `tests/pp_tests/<RUN_LABEL>/`.
3. For every parameter, record the **credible level of the truth**
   `p = mean(posterior < truth)`.
4. Make the **PP plot**: each parameter's empirical CDF of `p` should follow
   the diagonal if the pipeline is well-calibrated; a combined KS p-value is
   reported.

It is built from `Parameter_Estimation_ripplegw_injection.ipynb` and is
**Colab-compatible**, **fully configurable** (see the CONFIG cell), and
**stop-and-continue safe**: every injection checkpoints to disk, so re-running
the loop cell resumes from where it stopped (mount Google Drive for
persistence across Colab sessions).


## 1. Install GWgpu_jax (Colab only)

This cell is a no-op when GWgpu_jax is already importable (e.g. running locally
from the repo). On Colab it pins JAX to the version the CUDA plugin understands
and installs the package from GitHub.

**Auth:** provide a GitHub PAT via Colab Secrets (🔑 sidebar → add `GH_TOKEN`),
or you will be prompted. The token is scrubbed from the environment afterwards.


In [ ]:
import importlib.util, os

RUNNING_ON_COLAB = importlib.util.find_spec("google.colab") is not None
ALREADY_INSTALLED = importlib.util.find_spec("gwgpu_jax") is not None

if RUNNING_ON_COLAB and not ALREADY_INSTALLED:
    import getpass
    OWNER, REPO, BRANCH = "saulo-albuquerque-phys", "GWgpu_jax", "prior_definitions"

    # Pin JAX to the 0.4.x line Colab's CUDA plugin still supports.
    !pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
    !pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

    GH_TOKEN = None
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not GH_TOKEN:
        GH_TOKEN = getpass.getpass(f"GitHub PAT (for {OWNER}/{REPO}): ")
    os.environ["GH_TOKEN"] = GH_TOKEN

    !pip install -q "gwgpu_jax[data] @ git+https://$GH_TOKEN@github.com/{OWNER}/{REPO}.git@{BRANCH}"
    !pip install -q corner

    del os.environ["GH_TOKEN"]
    del GH_TOKEN
    print("Installed. If JAX was re-pinned, use Runtime → Restart session, then re-run from here.")
else:
    print("GWgpu_jax already importable — skipping install.")

In [ ]:
import time, json, glob
from pathlib import Path

import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Double precision is essential for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwgpu_jax
print("gwgpu_jax version :", gwgpu_jax.__version__)
print("JAX devices       :", jax.devices())

## 2. Configuration — all PE features live here

Everything you would normally tune is collected in this one cell:

* **`CONFIG`** — number of injections, output location, data segment, sampler
  budget, and seeds.
* **`PARAM_BOUNDS`** — the sampled parameters and their support (also the
  injection prior support).
* **`PRIORS`** — non-uniform prior shapes (`"sin"`, `"cos"`, `"volumetric"`);
  these are used **both** to draw the injections and inside the sampler, which
  is exactly what a PP test requires.

For a quick smoke test set `NUM_INJECTIONS = 3`, `NUM_LIVE = 300`,
`PHASE2_INNER_STEPS = 40`. The defaults below are tuned for a real validation
run and are correspondingly slow (≈ minutes per injection on a GPU).


In [ ]:
CONFIG = dict(
    # ── PP-test size ────────────────────────────────────────────────────
    NUM_INJECTIONS = 100,          # N injections drawn from the prior

    # ── Output location (stop-and-continue safe) ────────────────────────
    OUTPUT_ROOT = "tests/pp_tests", # results root (relative to repo, or absolute)
    RUN_LABEL   = "pptest_imrphenomd",  # sub-folder grouping this PP-test run
    MOUNT_DRIVE = False,           # Colab: mount Google Drive for persistence
    DRIVE_ROOT  = "/content/drive/MyDrive/gwjax_pp_tests",  # used iff MOUNT_DRIVE

    # ── Waveform / data segment ─────────────────────────────────────────
    APPROXIMANT   = "IMRPhenomD",
    F_REF         = 20.0,
    DETECTORS     = ["H1", "L1"],
    DURATION      = 4.0,           # s
    SAMPLING_RATE = 2048.0,        # Hz
    F_MIN         = 20.0,          # Hz
    F_MAX         = 512.0,         # Hz

    # ── Two-phase nested sampler budget ─────────────────────────────────
    NUM_LIVE           = 1000,
    PHASE1_INNER_STEPS = 40,       # bulk phase (cheap, vmap-amortised)
    PHASE2_INNER_STEPS = 120,      # accurate tail (main cost/quality lever)
    PHASE1_DELTA_LOGZ  = -1.0,
    PHASE1_MAX_ITERS   = 1000,
    PHASE2_MAX_ITERS   = 5000,
    LOG_DLOGZ_TARGET   = -3.0,
    NUM_POSTERIOR      = 2000,

    # ── Seeds (deterministic + resumable) ───────────────────────────────
    INJ_SEED       = 12345,        # draws the N injection parameter sets
    NOISE_SEED0    = 1000,         # noise seed for injection i = NOISE_SEED0 + i
    SAMPLER_SEED0  = 7000,         # sampler key for injection i = SAMPLER_SEED0 + i
)

# tc prior: inject_signal puts the merger at tc=0 in the segment, so the prior
# is a small window around 0 (synthetic-injection convention).
TC_CENTER, TC_HALFWIDTH = 0.0, 0.05

# Sampled parameters and their support. m1, m2 are sampled independently over
# the same range; IMRPhenomD's (m1,chi_1)<->(m2,chi_2) label symmetry is folded
# heavier-first for BOTH the truth and the posterior (see helpers), keeping the
# PP test self-consistent.
PARAM_BOUNDS = {
    "m1":          (10.0, 80.0),
    "m2":          (10.0, 80.0),
    "chi_1":       (-0.9,  0.9),
    "chi_2":       (-0.9,  0.9),
    "distance":    (50.0, 1500.0),
    "inclination": (0.0,  float(jnp.pi)),
    "ra":          (0.0,  2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0,  float(jnp.pi)),
    "phi_c":       (0.0,  2.0 * float(jnp.pi)),
    "tc":          (TC_CENTER - TC_HALFWIDTH, TC_CENTER + TC_HALFWIDTH),
}
FIXED_PARAMS = {}

# Non-uniform priors — used to draw injections AND inside the sampler.
PRIORS = {
    "inclination": "sin",          # p(theta) ∝ sin theta  (isotropic orientation)
    "dec":         "cos",          # p(dec)   ∝ cos dec     (isotropic sky)
    "distance":    "volumetric",   # p(d)     ∝ d^2         (uniform in volume → SNR spread)
}

PARAM_NAMES = list(PARAM_BOUNDS.keys())
print(f"sampling dimension : {len(PARAM_NAMES)}")
print(f"parameters         : {PARAM_NAMES}")

## 3. Output directory (and optional Google Drive)

Resolves where results are written. On Colab, set `MOUNT_DRIVE = True` in CONFIG
so the per-injection checkpoints survive a runtime restart — that is what makes
"stop and continue" work across sessions.


In [ ]:
if CONFIG["MOUNT_DRIVE"] and RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path(CONFIG["DRIVE_ROOT"])
else:
    output_root = Path(CONFIG["OUTPUT_ROOT"])

RUN_DIR = output_root / CONFIG["RUN_LABEL"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be written to:", RUN_DIR.resolve())

# Persist the configuration alongside the results for reproducibility.
with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"CONFIG": CONFIG, "PARAM_BOUNDS": PARAM_BOUNDS,
               "PRIORS": PRIORS, "FIXED_PARAMS": FIXED_PARAMS}, f, indent=2)

## 4. Draw the N injections from the prior

We draw all `NUM_INJECTIONS` parameter sets **once** from the prior with a fixed
seed and cache them to `injections.json`. On resume we reload that file, so the
exact same injections are always used. Truths are folded **heavier-first**
(`m1 ≥ m2`, carrying each spin with its mass) to match how the posterior is
folded later.


In [ ]:
def fold_heavier_first(p):
    """Return a copy with m1>=m2, carrying chi with its mass (dict of floats)."""
    p = dict(p)
    if p["m2"] > p["m1"]:
        p["m1"], p["m2"] = p["m2"], p["m1"]
        p["chi_1"], p["chi_2"] = p["chi_2"], p["chi_1"]
    return p

INJ_FILE = RUN_DIR / "injections.json"
if INJ_FILE.exists():
    injections = json.load(open(INJ_FILE))
    print(f"Loaded {len(injections)} cached injections from {INJ_FILE}")
else:
    prior_specs = gwgpu_jax.resolve_priors(PARAM_BOUNDS, PRIORS)
    particles, _ = gwgpu_jax.sample_prior(
        jax.random.PRNGKey(CONFIG["INJ_SEED"]), CONFIG["NUM_INJECTIONS"], prior_specs,
    )
    injections = []
    for i in range(CONFIG["NUM_INJECTIONS"]):
        truth = {name: float(np.asarray(particles[name])[i]) for name in PARAM_NAMES}
        truth.update(FIXED_PARAMS)
        injections.append(fold_heavier_first(truth))
    json.dump(injections, open(INJ_FILE, "w"), indent=2)
    print(f"Drew and cached {len(injections)} injections to {INJ_FILE}")

print("example injection [0]:",
      {k: round(v, 3) for k, v in injections[0].items()})

## 5. Per-injection PE (with checkpointing)

`run_one_injection` builds a fresh network, injects the IMRPhenomD signal into
Gaussian noise, runs the two-phase sampler, folds the posterior heavier-first,
saves everything into an ID-tagged folder, and returns the per-parameter
credible levels of the truth.

Each injection's folder (`inj_NNN/`) holds:
* `posterior_samples.npz` — folded posterior for every parameter,
* `corner.png` — corner plot with the truth overlaid,
* `result.json` — truth, network SNR, p-values, logZ, ESS, timing, and a
  `completed: true` flag used to skip the injection on resume.


In [ ]:
waveform_fn = gwgpu_jax.build_ripplegw_waveform_fn(
    CONFIG["APPROXIMANT"], f_ref=CONFIG["F_REF"],
)


def credible_levels(posterior, truth):
    """p = mean(posterior < truth) for each parameter (the PP-test statistic)."""
    return {n: float(np.mean(np.asarray(posterior[n]) < truth[n])) for n in PARAM_NAMES}


def _save_corner(posterior, truth, outfile):
    try:
        import corner
    except ImportError:
        return
    data = np.column_stack([np.asarray(posterior[n]) for n in PARAM_NAMES])
    fig = corner.corner(
        data, labels=PARAM_NAMES, truths=[truth[n] for n in PARAM_NAMES],
        range=[PARAM_BOUNDS[n] for n in PARAM_NAMES],
        quantiles=[0.16, 0.5, 0.84], show_titles=True, title_kwargs={"fontsize": 8},
    )
    fig.set_size_inches(13, 13)
    fig.savefig(outfile, dpi=90, bbox_inches="tight")
    plt.close(fig)


def run_one_injection(i, truth, *, force=False):
    """Run PE for injection i. Skips (loads) if already completed, unless force."""
    inj_dir = RUN_DIR / f"inj_{i:03d}"
    res_file = inj_dir / "result.json"
    if res_file.exists() and not force:
        rec = json.load(open(res_file))
        if rec.get("completed"):
            return rec, True   # (record, skipped)
    inj_dir.mkdir(parents=True, exist_ok=True)

    # 1. Fresh network + coloured Gaussian noise + IMRPhenomD injection.
    grid = gwgpu_jax.TimeFrequencyGrid(
        duration=CONFIG["DURATION"], sampling_rate=CONFIG["SAMPLING_RATE"],
        f_min=CONFIG["F_MIN"], f_max=CONFIG["F_MAX"],
    )
    network = gwgpu_jax.Network.from_names(CONFIG["DETECTORS"], grid)
    network.generate_noise(seed=CONFIG["NOISE_SEED0"] + i)

    hp, hc = waveform_fn(truth, grid.frequency_domain_array)
    h_dict = network.project_waveform(hp, hc, truth["ra"], truth["dec"], truth["psi"], gmst=0.0)
    net_snr = float(network.network_optimal_snr(h_dict))
    network.inject_signal(h_dict, domain="fd")

    # 2. Two-phase nested sampler with the SAME priors used to draw the truth.
    sampler = gwgpu_jax.GWgpu_jaxTwoPhaseNestedSampler(
        network=network, waveform_fn=waveform_fn,
        param_bounds=PARAM_BOUNDS, fixed_params=FIXED_PARAMS,
        priors=PRIORS, gmst=None,
    )
    t0 = time.perf_counter()
    result = sampler.run_two_phase(
        rng_key                     = jax.random.PRNGKey(CONFIG["SAMPLER_SEED0"] + i),
        num_live                    = CONFIG["NUM_LIVE"],
        phase1_num_inner_steps      = CONFIG["PHASE1_INNER_STEPS"],
        phase2_num_inner_steps      = CONFIG["PHASE2_INNER_STEPS"],
        phase1_num_delete           = max(1, CONFIG["NUM_LIVE"] // 20),
        phase1_delta_logz_threshold = CONFIG["PHASE1_DELTA_LOGZ"],
        phase1_max_iterations       = CONFIG["PHASE1_MAX_ITERS"],
        phase2_num_delete           = 1,
        phase2_max_iterations       = CONFIG["PHASE2_MAX_ITERS"],
        log_dlogz_target            = CONFIG["LOG_DLOGZ_TARGET"],
        num_posterior_samples       = CONFIG["NUM_POSTERIOR"],
        verbose                     = False,
    )
    elapsed = time.perf_counter() - t0

    # 3. Fold posterior heavier-first (matches the folded truth).
    post = {n: np.asarray(result.posterior_samples[n]) for n in PARAM_NAMES}
    swap = post["m2"] > post["m1"]
    post["m1"], post["m2"] = np.where(swap, post["m2"], post["m1"]), np.where(swap, post["m1"], post["m2"])
    post["chi_1"], post["chi_2"] = np.where(swap, post["chi_2"], post["chi_1"]), np.where(swap, post["chi_1"], post["chi_2"])

    # 4. Save posterior, corner plot, and the result record.
    np.savez_compressed(inj_dir / "posterior_samples.npz", **post)
    _save_corner(post, truth, inj_dir / "corner.png")

    p_levels = credible_levels(post, truth)
    rec = dict(
        injection_id=f"inj_{i:03d}", index=i, completed=True,
        truth=truth, network_snr=net_snr, p_values=p_levels,
        logZ=float(result.logZ), logZ_err=float(result.logZ_err),
        ess=float(result.ess), n_iterations=int(result.n_iterations),
        elapsed_s=elapsed,
    )
    json.dump(rec, open(res_file, "w"), indent=2)
    return rec, False

## 6. Run the PP-test loop — stop and continue any time

Run this cell to process all injections. Already-completed injections are
skipped instantly (loaded from disk), so you can **interrupt the kernel and
re-run this cell** to continue. A `manifest.csv` summarising every completed
injection (SNR, ESS, logZ) is refreshed at the end.


In [ ]:
import csv

records = []
n_done, n_skipped = 0, 0
for i in range(CONFIG["NUM_INJECTIONS"]):
    truth = injections[i]
    rec, skipped = run_one_injection(i, truth)
    records.append(rec)
    if skipped:
        n_skipped += 1
    else:
        n_done += 1
        print(f"[{i+1:3d}/{CONFIG['NUM_INJECTIONS']}] inj_{i:03d}  "
              f"SNR={rec['network_snr']:5.1f}  ESS={rec['ess']:6.1f}  "
              f"logZ={rec['logZ']:+8.1f}  ({rec['elapsed_s']:.0f}s)")

print(f"\nDone. {n_done} newly run, {n_skipped} skipped (already complete).")

# Refresh the manifest from every completed result.json.
manifest_rows = []
for rf in sorted(glob.glob(str(RUN_DIR / "inj_*" / "result.json"))):
    r = json.load(open(rf))
    if r.get("completed"):
        manifest_rows.append(dict(
            injection_id=r["injection_id"], network_snr=r["network_snr"],
            ess=r["ess"], logZ=r["logZ"], n_iterations=r["n_iterations"],
            elapsed_s=r.get("elapsed_s", float("nan")),
        ))
if manifest_rows:
    with open(RUN_DIR / "manifest.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
        w.writeheader(); w.writerows(manifest_rows)
    snrs = [m["network_snr"] for m in manifest_rows]
    print(f"manifest.csv: {len(manifest_rows)} injections, "
          f"SNR range {min(snrs):.1f}–{max(snrs):.1f}")

## 7. PP plot — validate the pipeline

Gathers the credible levels from every completed injection and plots, for each
parameter, the empirical CDF of `p` against the diagonal. A well-calibrated
pipeline produces curves that stay inside the grey confidence bands. The legend
shows each parameter's KS p-value against Uniform(0,1); the title shows the
**combined** p-value (Fisher's method) — values that are not tiny indicate the
ensemble is consistent with perfect calibration.


In [ ]:
from scipy import stats

# Collect p-values from all completed injections.
all_recs = [json.load(open(rf)) for rf in
            sorted(glob.glob(str(RUN_DIR / "inj_*" / "result.json")))]
all_recs = [r for r in all_recs if r.get("completed")]
N = len(all_recs)
assert N > 0, "No completed injections found — run the loop cell first."
print(f"PP plot from {N} completed injections.")

pp = {n: np.sort([r["p_values"][n] for r in all_recs]) for n in PARAM_NAMES}
x = np.linspace(0, 1, 200)

fig, ax = plt.subplots(figsize=(8, 8))
# 1/2/3-sigma confidence bands for the empirical CDF under the null (binomial).
for sig, alpha in [(1, 0.3), (2, 0.2), (3, 0.1)]:
    edge = stats.norm.cdf(sig)
    lo = stats.binom.ppf(1 - edge, N, x) / N
    hi = stats.binom.ppf(edge, N, x) / N
    ax.fill_between(x, lo, hi, color="0.5", alpha=alpha, lw=0)
ax.plot([0, 1], [0, 1], "k--", lw=1)

ks_p = {}
for n in PARAM_NAMES:
    emp = np.searchsorted(pp[n], x, side="right") / N
    ks_p[n] = stats.kstest(pp[n], "uniform").pvalue
    ax.plot(x, emp, lw=1.5, label=f"{n} (p={ks_p[n]:.2f})")

# Combined p-value across parameters (Fisher's method).
combined = stats.combine_pvalues(list(ks_p.values()), method="fisher").pvalue
ax.set_xlabel("credible level  p"); ax.set_ylabel("fraction of injections < p")
ax.set_title(f"PP test — N={N} injections, combined p = {combined:.3f}")
ax.legend(fontsize=8, loc="upper left", ncol=2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(RUN_DIR / "pp_plot.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved", RUN_DIR / "pp_plot.png")
print("per-parameter KS p-values:", {k: round(v, 3) for k, v in ks_p.items()})

## Notes

- **Stop & continue:** interrupt the loop cell whenever; re-running it skips
  completed injections (`result.json` with `completed: true`). On Colab set
  `MOUNT_DRIVE = True` so checkpoints survive a runtime restart.
- **Different SNRs:** the volumetric distance prior gives a broad SNR spread;
  see `manifest.csv` for the per-injection network SNR.
- **Cost vs. quality:** `PHASE2_INNER_STEPS` and `NUM_LIVE` are the main levers.
  For a quick check drop `NUM_INJECTIONS`, `NUM_LIVE`, and `PHASE2_INNER_STEPS`.
- **Interpretation:** curves inside the bands and a non-tiny combined p-value ⇒
  the sampler + priors are well-calibrated. A consistent diagonal offset for one
  parameter points to a bias in that parameter's prior/likelihood handling.
